# 06f — Intermittent Demand Modeling

Handles the **Intermittent** regime from 06c's Syntetos-Boylan classification — the
SKUs routed to a Croston-family method rather than the Tweedie + conformal pipeline
in 06d/06e, since these are mostly-zero series where a residual-quantile approach
breaks down.

**Scope note:** Lumpy SKUs are deliberately *out of scope* here. 06c already routed
them to a fixed 'policy' rather than any demand model (lumpy demand is close to
unforecastable in both timing and size — a conservative safety-stock rule is more
robust than a fitted model). Lumpy's policy gets defined in 06g as a simulation-time
rule, not built as a model in this notebook.

**Pipeline for this notebook:**
1. Isolate the Intermittent SKUs and confirm regime/routing counts
2. Implement TSB (Teunter-Syntetos-Babai) for point forecasts
3. Bootstrap-based lead-time-demand simulation for the service-level/safety-stock question
4. Save outputs in the same format 06e used, so 06g can load both regimes uniformly


## Section 1 — Setup & Regime Isolation

**Why TSB over Croston's Method or SBA:**

Croston's Method smooths non-zero demand size and inter-demand interval
independently, and updates *either* estimate only when a demand event occurs.
Syntetos & Boylan (2001) showed this produces a systematic positive bias — the
ratio of two separately-smoothed, non-independent estimators isn't itself an
unbiased estimator of the mean. SBA patches that one bias with a correction
factor `(1 - α/2)`.

Neither method addresses a second, more operationally dangerous failure mode:
because both only update on a non-zero observation, a SKU that goes fully
obsolete keeps forecasting its old demand level indefinitely — there's no decay
mechanism. TSB (Teunter-Syntetos-Babai, 2011) fixes this by smoothing the
*probability of demand occurring* every period, zero-periods included, so a long
enough run of zeros pulls the forecast toward zero on its own.

Given this project's stated framing — flagging stockout risk before it becomes a
supply chain failure — the failure mode TSB protects against (continuing to
reorder for a dead SKU) is at least as costly as the one Croston/SBA protect
against. TSB costs nothing extra to implement (still two exponential-smoothing
recursions, no added hyperparameters), so this isn't a close tradeoff — it
dominates at zero added cost.


In [5]:
# ── Path config (each notebook file redefines this locally -- no cross-kernel carryover) ─
PROCESSED_DIR = '../data/processed'
RAW_DIR = '../data/raw'

# ── 06f Section 1: Setup — isolate the Intermittent SKUs 06d/06e skipped ─────────
import pandas as pd
import numpy as np
import pickle
import os

sku_regimes = pd.read_parquet(f'{PROCESSED_DIR}/segmentation/sku_regimes_fold2.parquet')

with open(f'{PROCESSED_DIR}/segmentation/regime_thresholds.pkl', 'rb') as f:
    regime_thresholds = pickle.load(f)

print('── Regime thresholds used for classification (from 06c) ──')
print(regime_thresholds)
print()

print('── Full regime breakdown (all 4 quadrants from 06c) ──')
print(sku_regimes['regime'].value_counts())
print()

# 06c already routed each regime to a modeling approach -- Lumpy -> 'policy', not a demand
# model. That's a deliberate, standard call (lumpy demand is close to unforecastable in
# both timing and size; a simple conservative safety-stock rule is more robust than a
# fitted model here), so 06f respects that routing rather than overriding it. This
# notebook builds TSB for the 'croston'-routed Intermittent regime ONLY. Lumpy's policy
# rule is real work but belongs in 06g as a simulation-time rule, not a fitted model here.
routing = regime_thresholds['routing_counts']
print(f'Routing from 06c: {routing}')
print(f"06f scope:              Intermittent only ({routing['croston']:,} SKUs, Croston-family/TSB)")
print(f"Out of scope here:      Lumpy ({routing['policy']:,} SKUs, policy-based -- to be defined in 06g)")
print(f"Already handled 06d/06e: Smooth+Erratic ({routing['tweedie']:,} SKUs, Tweedie+conformal)")
print()

modeling_ids = sku_regimes.loc[sku_regimes['regime'] == 'intermittent', 'id'].tolist()
print(f'Intermittent SKUs entering 06f: {len(modeling_ids):,}')
print()

# ── Zero-fraction sanity check, straight from the raw wide-format file ──────────────────
# Filtering to just the Intermittent SKUs first keeps this light -- no need to touch the
# 440MB/97MB feature parquets for a sanity check this simple.
raw_sales = pd.read_csv(f'{RAW_DIR}/sales_train_validation.csv')
day_cols = [c for c in raw_sales.columns if c.startswith('d_')]

intermittent_sales = raw_sales[raw_sales['id'].isin(modeling_ids)].set_index('id')[day_cols]
zero_frac = (intermittent_sales == 0).mean(axis=1)

print(f'Intermittent SKUs found in raw data: {len(intermittent_sales):,} (expect {len(modeling_ids):,})')
print()
print('── Zero-fraction of daily demand, Intermittent SKUs ──')
print(zero_frac.describe().round(3))

── Regime thresholds used for classification (from 06c) ──
{'adi_threshold': 1.32, 'cv2_threshold': 0.49, 'aggregation': 'weekly', 'fold': 'fold2', 'train_end': '2014-01-31', 'n_skus_classified': 30490, 'regime_counts': {'intermittent': 14268, 'smooth': 8389, 'lumpy': 7003, 'erratic': 830}, 'routing_counts': {'croston': 14268, 'tweedie': 9219, 'policy': 7003}}

── Full regime breakdown (all 4 quadrants from 06c) ──
regime
intermittent    14268
smooth           8389
lumpy            7003
erratic           830
Name: count, dtype: int64

Routing from 06c: {'croston': 14268, 'tweedie': 9219, 'policy': 7003}
06f scope:              Intermittent only (14,268 SKUs, Croston-family/TSB)
Out of scope here:      Lumpy (7,003 SKUs, policy-based -- to be defined in 06g)
Already handled 06d/06e: Smooth+Erratic (9,219 SKUs, Tweedie+conformal)

Intermittent SKUs entering 06f: 14,268

Intermittent SKUs found in raw data: 14,268 (expect 14,268)

── Zero-fraction of daily demand, Intermittent SKUs ──
cou

### Section 1 Findings — Regime Isolation & Scope Confirmation

- **ID match: 14,268 of 14,268 (100%)** — every SKU that 06c classified as Intermittent
  was found in the raw file under the same `id` format. No join/format mismatch carried
  forward into modeling, which was worth checking explicitly rather than assuming.
- **Zero-fraction distribution:** mean 74.2%, median 77.9%, IQR 63.0%–87.8%, range
  17.3%–99.1%. Cross-checked against 01_eda's overall M5 zero-inflation baseline
  (68.2% across all 30,490 series) — Intermittent SKUs sit meaningfully above that
  baseline, which is independent (daily-level, computed directly from raw counts)
  confirmation that 06c's classifier is picking out genuinely sparser series. Note this
  is *directional* support, not a literal same-statistic validation: the classifier's
  ADI/CV² cutoffs were computed on **weekly**-aggregated demand intervals
  (`aggregation: 'weekly'` in `regime_thresholds`), while this zero-fraction check is on
  **daily** data — different units, consistent conclusion.
- **Wide within-regime spread (17.3%–99.1%):** "Intermittent" is not a homogeneous
  group — a SKU with 17% zero-days is barely sparser than a typical Erratic SKU, while
  one at 99.1% almost never sells. A single fixed smoothing parameter applied uniformly
  across all 14,268 SKUs risks under-reacting for the sparsest tail and over-reacting
  for the densest one — flagged here since it directly shapes the TSB parameter choice
  in Section 2.

**Scope confirmed:** 06f models Intermittent (14,268 SKUs) only. Lumpy (7,003) stays
routed to a fixed policy per 06c, to be defined in 06g.

### Section 2 — TSB Implementation:

The core design call here: with often only a few dozen non-zero observations per SKU (recall the sparsest SKUs are ~99% zero over ~3 years of daily data), fitting bespoke (alpha, beta) per SKU via optimization is asking for overfit noise, not signal — there isn't enough data per series to trust an individually-tuned smoothing parameter. Standard practice (Teunter et al. 2011; Boylan & Syntetos) is to start from literature-default smoothing values shared across the whole regime, confirm the mechanism behaves sensibly, and only reach for optimization later if there's evidence a single shared value is actually costing accuracy — not before. This section builds and sanity-checks the mechanism; parameter selection is Section 3's job, done properly (walk-forward, not in-sample).

In [6]:
# ── 06f Section 2: TSB (Teunter-Syntetos-Babai) implementation ──────────────────────────

def tsb_forecast(demand, alpha=0.1, beta=0.1):
    """
    One-step-ahead TSB forecast for a single SKU's demand series.

    p: smoothed probability that demand occurs -- updated EVERY period (zero periods
       included). This is the mechanism that lets TSB decay toward zero for a SKU that's
       gone obsolete, which Croston/SBA cannot do (they only update at demand events).
    z: smoothed non-zero demand size -- updated ONLY when demand actually occurs, same
       as Croston/SBA.
    forecast = p * z, the expected demand per period.
    """
    n = len(demand)
    p = np.zeros(n)
    z = np.zeros(n)
    forecast = np.zeros(n)

    nonzero_mask = demand > 0
    if not nonzero_mask.any():
        return forecast, 0.0, 0.0  # SKU with zero sales in this window -- forecast stays 0

    first_nonzero_idx = np.argmax(nonzero_mask)
    z[0] = demand[first_nonzero_idx]
    p[0] = nonzero_mask.mean()  # crude initial probability estimate from full window
    forecast[0] = p[0] * z[0]

    for t in range(1, n):
        occurred = demand[t - 1] > 0
        p[t] = beta * occurred + (1 - beta) * p[t - 1]
        z[t] = alpha * demand[t - 1] + (1 - alpha) * z[t - 1] if occurred else z[t - 1]
        forecast[t] = p[t] * z[t]

    return forecast, p[-1], z[-1]


# ── Sanity check: fixed default params (alpha=beta=0.1), all 14,268 Intermittent SKUs ───
# Not tuned yet -- this is a mechanism check before Section 3's proper parameter selection.
ALPHA_DEFAULT, BETA_DEFAULT = 0.1, 0.1

demand_matrix = intermittent_sales.values  # from Section 1: rows=SKU, cols=day, in id order
sku_ids_ordered = intermittent_sales.index.tolist()

mae_scores = []
final_states = {}

for i, sku_id in enumerate(sku_ids_ordered):
    demand = demand_matrix[i]
    forecast, p_final, z_final = tsb_forecast(demand, ALPHA_DEFAULT, BETA_DEFAULT)

    # in-sample one-step MAE, vs. naive "always predict the SKU's own mean demand"
    naive_mae = np.abs(demand - demand.mean()).mean()
    tsb_mae = np.abs(demand[1:] - forecast[1:]).mean()  # skip index 0 (init period)

    mae_scores.append({'id': sku_id, 'tsb_mae': tsb_mae, 'naive_mae': naive_mae})
    final_states[sku_id] = {'p_final': p_final, 'z_final': z_final}

mae_df = pd.DataFrame(mae_scores)
mae_df['tsb_beats_naive'] = mae_df['tsb_mae'] < mae_df['naive_mae']

print(f'TSB beats naive-mean baseline on {mae_df["tsb_beats_naive"].mean()*100:.1f}% of Intermittent SKUs')
print()
print('── MAE comparison, TSB (default params) vs. naive mean ──')
print(mae_df[['tsb_mae', 'naive_mae']].describe().round(3))

TSB beats naive-mean baseline on 98.5% of Intermittent SKUs

── MAE comparison, TSB (default params) vs. naive mean ──
         tsb_mae  naive_mae
count  14268.000  14268.000
mean       0.600      0.882
std        0.725      1.319
min        0.019      0.021
25%        0.239      0.267
50%        0.404      0.508
75%        0.684      0.955
max       15.652     28.224


### Section 2 Findings — TSB Mechanism Check

- **TSB (default params, alpha=beta=0.1) beats naive-mean on 98.5% of Intermittent
  SKUs** (14,053 of 14,268) — mean MAE 0.600 vs. 0.882 (~32% reduction), median 0.404
  vs. 0.508 (~20% reduction). Confirms the probability × size functional form is
  capturing real signal, not just adding complexity for nothing.
- **Caveat, by design:** this was in-sample, over the SKU's full raw history, with a
  single un-tuned default parameter pair applied uniformly. That's enough to confirm
  the mechanism works — not enough to select a parameter value or claim forecast skill
  on unseen data. Section 3 addresses both: split on the pipeline's actual fold2
  boundary (`train_end='2014-01-31'`, from `regime_thresholds`) via `calendar.csv`,
  not an arbitrary new split, and evaluate strictly on the held-out period.
- **The 1.5% (215 SKUs) where TSB doesn't beat naive** are worth a quick look before
  finalizing — likely near-constant low-volume SKUs where a global mean already
  captures almost everything. Flagged for a follow-up check, not blocking Section 3.

### Section 3 — Parameter Selection (Walk-Forward, Fold2 Split)

The design call here: per-SKU tuned (alpha, beta) is off the table for the same reason it was in Section 2 — many of these series have only a few dozen non-zero observations, nowhere near enough to trust an individually-fit smoothing parameter. So this mirrors 06e's cost-ratio sweep philosophy exactly: grid-search a small, literature-grounded range (Teunter et al. recommend low constants for intermittent demand, typically 0.05–0.3 — values much above that overreact to single sporadic spikes) and pick by aggregate held-out error, rather than optimizing a single continuous value off data too thin to support it.

One more efficiency call worth stating explicitly: grid-searching all 14,268 SKUs × 9 parameter combos is real computation for not much benefit — an aggregate MAE over a 3,000-SKU random sample is already a stable enough estimate to rank 9 candidates. The full population only gets scored once, on the winning pair, at the end. That's the same "don't pay for precision you don't need at the exploration stage" logic as 06e's sensitivity sweep.

In [7]:
# ── Section 3: Parameter Selection (Walk-Forward, matching the fold2 train/val split) ──
# Sparse per-SKU history rules out per-SKU tuned parameters -- same logic as 06e's cost-
# ratio sweep: don't optimize a continuous point off data too thin to trust it. Grid
# search runs on a random subsample for speed; the full 14,268 only get scored once,
# on the winning pair, at the end.

calendar = pd.read_csv(f'{RAW_DIR}/calendar.csv')
calendar['date'] = pd.to_datetime(calendar['date'])
train_end_date = pd.to_datetime(regime_thresholds['train_end'])

train_days = set(calendar.loc[calendar['date'] <= train_end_date, 'd'])
split_idx = sum(1 for d in day_cols if d in train_days)

print(f'Train days: {split_idx:,} (through {train_end_date.date()})')
print(f'Val days:   {len(day_cols) - split_idx:,}')
print()

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(demand_matrix), size=min(3000, len(demand_matrix)), replace=False)
sample_matrix = demand_matrix[sample_idx]

ALPHA_GRID = [0.05, 0.1, 0.2]
BETA_GRID  = [0.05, 0.1, 0.2]

grid_results = []
for alpha in ALPHA_GRID:
    for beta in BETA_GRID:
        val_errors = [
            np.abs(demand[split_idx:] - tsb_forecast(demand, alpha, beta)[0][split_idx:]).mean()
            for demand in sample_matrix
        ]
        grid_results.append({'alpha': alpha, 'beta': beta, 'val_mae_sample': np.mean(val_errors)})

grid_df = pd.DataFrame(grid_results).sort_values('val_mae_sample').reset_index(drop=True)
print(f'── Grid search (n={len(sample_matrix):,} SKU sample, held-out val period) ──')
print(grid_df.round(4).to_string(index=False))
print()

ALPHA_BEST, BETA_BEST = grid_df.loc[0, 'alpha'], grid_df.loc[0, 'beta']
print(f'Selected: alpha={ALPHA_BEST}, beta={BETA_BEST}')
print()

# ── Confirm on the FULL 14,268 SKUs, winning params only ────────────────────────────────
full_val_errors = []
full_forecasts = {}
for sku_id, demand in zip(sku_ids_ordered, demand_matrix):
    forecast, p_final, z_final = tsb_forecast(demand, ALPHA_BEST, BETA_BEST)
    full_val_errors.append(np.abs(demand[split_idx:] - forecast[split_idx:]).mean())
    full_forecasts[sku_id] = {'forecast': forecast, 'p_final': p_final, 'z_final': z_final}

print(f'Full-population held-out MAE, winning params: {np.mean(full_val_errors):.4f}')
print(f'(sample estimate was: {grid_df.loc[0, "val_mae_sample"]:.4f} -- should be close if the sample was representative)')

Train days: 1,099 (through 2014-01-31)
Val days:   814

── Grid search (n=3,000 SKU sample, held-out val period) ──
 alpha  beta  val_mae_sample
  0.10  0.20          0.7984
  0.05  0.20          0.8001
  0.20  0.20          0.8011
  0.10  0.10          0.8068
  0.05  0.10          0.8089
  0.20  0.10          0.8090
  0.10  0.05          0.8210
  0.20  0.05          0.8226
  0.05  0.05          0.8236

Selected: alpha=0.1, beta=0.2

Full-population held-out MAE, winning params: 0.7723
(sample estimate was: 0.7984 -- should be close if the sample was representative)


### First of all, is section 2 and 3 correct like how a princial data scientist would do it. And does it follow what our new plan says. And if iit doesnt, we need to change our new plan, or if our new_plan is better, then follow that instead. We need to make sure we are doing this all properly. Like why are we using MAE as an eval metric. Why is the MAE so high, do we need to do different grid search or larger sample, this only took 2 minutes to run. Something isnt right. Is all of this correct, etc. 